# 10 — Evaluation: strategi, portofolio, rezim, biaya, dan ketidakpastian

Notebook ini menyatukan evaluasi strategi trading dan portofolio optimasi pada satu kerangka out-of-sample. Semua strategi memakai biaya transaksi 0,15% beli, 0,25% jual, dan slippage 0,05% pada skenario 1x; parameter biaya dapat diuji pada 0x, 1x, dan 2x.

Kesimpulan akhir tidak mengasumsikan strategi pasti unggul. Sebuah strategi disebut `strict beat` hanya jika CAGR dan Sharpe net-nya mengungguli kedua benchmark: LQ45 dan equal-weight 15 saham.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.strategy import ma_crossover_50_200, mean_reversion_bollinger_rsi, momentum_12_1_monthly
from src.backtester import BacktestConfig, backtest

warnings.filterwarnings('ignore', category=FutureWarning)
TRADING_DAYS_PER_YEAR = 252
COST_MULTIPLES = (0.0, 1.0, 2.0)
RANDOM_STATE = 42

## Data dan signal matrix

Strategi pada `src.strategy` sudah mengembalikan bobot yang dieksekusi pada hari berikutnya. Backtester menerima sinyal sebelum eksekusi, sehingga bobot tersebut dibalik satu baris terlebih dahulu agar tidak terkena execution lag dua kali.

In [2]:
master = pd.read_parquet(ROOT / 'data' / 'processed' / 'master_daily.parquet')
master['date'] = pd.to_datetime(master['date'])
master = master.sort_values('date').reset_index(drop=True)
prices = master[['date'] + [c for c in master.columns if str(c).endswith('.JK')]].copy()
stocks = [c for c in prices.columns if str(c).endswith('.JK')]
assert len(stocks) == 15
bi_rate_annual = pd.to_numeric(master.set_index('date')['bi_rate'], errors='coerce').ffill() / 100.0

def executed_to_signal(executed_weights):
    """Undo the one-row execution shift for src.strategy output."""
    signal = executed_weights.copy().shift(-1).fillna(0.0)
    signal.index.name = 'date'
    return signal[stocks]

strategy_executed = {
    'MA_50_200': ma_crossover_50_200(prices),
    'MeanReversion_BB_RSI': mean_reversion_bollinger_rsi(prices),
    'Momentum_12_1': momentum_12_1_monthly(prices),
}
strategy_signals = {name: executed_to_signal(frame) for name, frame in strategy_executed.items()}
print({name: frame.shape for name, frame in strategy_signals.items()})

{'MA_50_200': (2410, 15), 'MeanReversion_BB_RSI': (2410, 15), 'Momentum_12_1': (2410, 15)}


### Tambahan ML dan portofolio optimasi

ML signal dari notebook 07 dipetakan menjadi strategi market timing: jika model memprediksi LQ45 naik, seluruh 15 saham diberi equal-weight; jika tidak, portofolio berada di cash. Bobot portofolio dari notebook 09 dibaca sebagai sinyal pada tanggal rebalance dan di-forward-fill sampai rebalance berikutnya.

In [3]:
optional_notes = []

ml_path = ROOT / 'data' / 'processed' / 'ml_signal_predictions.csv'
if ml_path.exists():
    ml = pd.read_csv(ml_path, parse_dates=['date'])
    for model_name, group in ml.groupby('model'):
        signal = pd.DataFrame(0.0, index=pd.DatetimeIndex(prices['date']), columns=stocks)
        up_dates = pd.DatetimeIndex(group.loc[group['prediction'].eq(1), 'date'])
        signal.loc[signal.index.intersection(up_dates)] = 1.0
        signal.index.name = 'date'
        strategy_signals[f'ML_{model_name}'] = signal
else:
    optional_notes.append('ML predictions tidak ditemukan; jalankan 07_ml_signal.ipynb.')

portfolio_path = ROOT / 'reports' / 'portfolio_optimization_weights.csv'
if portfolio_path.exists():
    portfolio_raw = pd.read_csv(portfolio_path, header=[0, 1], index_col=0, parse_dates=True)
    portfolio_raw.index.name = 'date'
    portfolio_methods = list(portfolio_raw.columns.get_level_values(0).unique())
    for method in portfolio_methods:
        period_weights = portfolio_raw[method].reindex(columns=stocks).astype(float)
        schedule = pd.DataFrame(np.nan, index=pd.DatetimeIndex(prices['date']), columns=stocks)
        for date, row in period_weights.iterrows():
            if date in schedule.index:
                schedule.loc[date] = row.to_numpy()
        strategy_signals[f'Portfolio_{method}'] = schedule.ffill().fillna(0.0)
else:
    optional_notes.append('Bobot portfolio belum ditemukan; jalankan 09_portfolio_optimization.ipynb.')

print('Candidates:', list(strategy_signals))
if optional_notes:
    print('Catatan:', ' '.join(optional_notes))

Candidates: ['MA_50_200', 'MeanReversion_BB_RSI', 'Momentum_12_1', 'ML_logistic_regression', 'ML_naive_majority', 'ML_random_forest', 'ML_xgboost', 'Portfolio_max_sharpe', 'Portfolio_min_volatility', 'Portfolio_hrp']


## Backtest net biaya

Benchmark pada setiap hasil backtest bersifat gross/no-cost. Strategi dan portofolio memakai biaya sesuai skenario. Ini konservatif untuk menilai apakah strategi net benar-benar dapat mengungguli benchmark pasar yang tidak melakukan turnover aktif.

In [4]:
BASE_CONFIG = BacktestConfig()

def config_for_multiple(multiple):
    return BacktestConfig(
        initial_capital=BASE_CONFIG.initial_capital,
        buy_fee=BASE_CONFIG.buy_fee * multiple,
        sell_fee=BASE_CONFIG.sell_fee * multiple,
        slippage=BASE_CONFIG.slippage * multiple,
        execution='close',
    )

def run_candidates(candidates, cost_multiple=1.0):
    results = {}
    for name, signal in candidates.items():
        results[name] = backtest(
            signals=signal, prices=prices, master_daily=master,
            config=config_for_multiple(cost_multiple),
        )
    return results

base_results = run_candidates(strategy_signals, cost_multiple=1.0)

def metrics_table(results, cost_multiple=1.0):
    rows = []
    for name, result in results.items():
        row = result.metrics.loc['strategy'].to_dict()
        row.update({'name': name, 'category': 'strategy_or_portfolio', 'cost_multiple': cost_multiple})
        rows.append(row)
    first = next(iter(results.values()))
    for benchmark in ['benchmark_lq45', 'benchmark_equal_weight']:
        row = first.metrics.loc[benchmark].to_dict()
        row.update({'name': benchmark, 'category': 'benchmark', 'cost_multiple': 0.0})
        rows.append(row)
    return pd.DataFrame(rows).set_index('name')

all_metrics = metrics_table(base_results, cost_multiple=1.0)
all_metrics.style.format({
    'CAGR': '{:.2%}', 'volatility_annual': '{:.2%}', 'Sharpe': '{:.2f}',
    'Sortino': '{:.2f}', 'Max Drawdown': '{:.2%}', 'Calmar': '{:.2f}',
    'win_rate': '{:.2%}', 'turnover_annual': '{:.2f}x',
})

,CAGR,volatility_annual,Sharpe,Sortino,Max Drawdown,Calmar,win_rate,turnover_annual,category,cost_multiple
name,,,,,,,,,,
MA_50_200,3.84%,19.62%,0.04,0.06,-47.00%,0.08,50.65%,8.79x,strategy_or_portfolio,1.000000
MeanReversion_BB_RSI,0.50%,27.10%,-0.02,-0.03,-55.92%,0.01,47.85%,28.85x,strategy_or_portfolio,1.000000
Momentum_12_1,3.16%,22.77%,0.04,0.06,-50.33%,0.06,49.91%,7.85x,strategy_or_portfolio,1.000000
ML_logistic_regression,-1.05%,11.63%,-0.45,-0.66,-40.78%,-0.03,47.64%,9.40x,strategy_or_portfolio,1.000000
ML_naive_majority,5.91%,17.34%,0.14,0.20,-44.08%,0.13,51.65%,2.51x,strategy_or_portfolio,1.000000
ML_random_forest,0.11%,13.78%,-0.27,-0.40,-38.96%,0.00,47.02%,11.52x,strategy_or_portfolio,1.000000
ML_xgboost,-3.07%,13.48%,-0.52,-0.74,-42.88%,-0.07,44.04%,15.98x,strategy_or_portfolio,1.000000
Portfolio_max_sharpe,-0.14%,20.14%,-0.15,-0.21,-42.73%,-0.00,49.44%,2.79x,strategy_or_portfolio,1.000000
Portfolio_min_volatility,2.82%,15.98%,-0.05,-0.07,-37.36%,0.08,50.00%,2.28x,strategy_or_portfolio,1.000000


## Performa berdasarkan rezim

Rezim yang dipakai: `2020_krisis`, `2022_kenaikan_suku_bunga`, dan `2025_2026`. BI Rate dan PMI-BI ditampilkan sebagai marker validasi rezim, bukan sebagai fitur yang melihat masa depan. Hari di luar tiga jendela tersebut diberi label `lainnya`.

In [5]:
macro = master.set_index('date')[['bi_rate', 'pmi_bi']].apply(pd.to_numeric, errors='coerce').ffill()
regime_info = pd.DataFrame(index=macro.index)
regime_info['bi_rate'] = macro['bi_rate']
regime_info['pmi_bi'] = macro['pmi_bi']
regime_info['bi_rate_change'] = macro['bi_rate'].diff()
regime_info['pmi_expansion'] = macro['pmi_bi'] > 50
regime_info['regime'] = 'lainnya'
regime_info.loc[regime_info.index.year == 2020, 'regime'] = '2020_krisis'
regime_info.loc[regime_info.index.year == 2022, 'regime'] = '2022_kenaikan_suku_bunga'
regime_info.loc[regime_info.index >= '2025-01-01', 'regime'] = '2025_2026'

regime_markers = regime_info.groupby('regime').agg(
    start=('bi_rate', 'first'), end=('bi_rate', 'last'),
    average_bi_rate=('bi_rate', 'mean'), average_pmi=('pmi_bi', 'mean'),
    pct_pmi_expansion=('pmi_expansion', 'mean'),
).reindex(['2020_krisis', '2022_kenaikan_suku_bunga', '2025_2026', 'lainnya'])
regime_markers.style.format({
    'start': '{:.2f}', 'end': '{:.2f}', 'average_bi_rate': '{:.2f}',
    'average_pmi': '{:.2f}', 'pct_pmi_expansion': '{:.2%}',
})

,start,end,average_bi_rate,average_pmi,pct_pmi_expansion
regime,,,,,
2020_krisis,5.00,3.75,4.32,42.72,27.69%
2022_kenaikan_suku_bunga,3.50,5.50,3.92,52.29,98.37%
2025_2026,6.00,5.75,5.26,51.58,100.00%
lainnya,5.00,6.00,5.09,51.15,80.16%


In [6]:
def regime_metrics(result, regime_info):
    curve = result.equity_curve.copy()
    curve['regime'] = regime_info['regime'].reindex(curve.index).ffill()
    rows = []
    for regime, group in curve.groupby('regime'):
        returns = group['daily_return']
        equity = (1.0 + returns).cumprod()
        rf = group['risk_free_daily']
        excess = returns - rf
        downside = np.minimum(excess.to_numpy(), 0.0)
        dd = equity / equity.cummax() - 1.0
        rows.append({
            'name': result.equity_curve.attrs.get('name', 'strategy'),
            'regime': regime, 'days': len(group),
            'return': float(equity.iloc[-1] - 1.0),
            'volatility_annual': float(returns.std(ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)),
            'sharpe': float(excess.mean() / excess.std(ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)) if excess.std(ddof=1) > 0 else np.nan,
            'max_drawdown': float(dd.min()),
            'win_rate': float((returns > 0).mean()),
        })
    return pd.DataFrame(rows)

regime_rows = []
for name, result in base_results.items():
    curve = result.equity_curve.copy()
    curve.attrs['name'] = name
    result.equity_curve = curve
    regime_rows.append(regime_metrics(result, regime_info))
regime_table = pd.concat(regime_rows, ignore_index=True)
regime_table = regime_table.merge(regime_markers.reset_index(), on='regime', how='left')
regime_table.sort_values(['regime', 'return'], ascending=[True, False]).style.format({
    'return': '{:.2%}', 'volatility_annual': '{:.2%}', 'sharpe': '{:.2f}',
    'max_drawdown': '{:.2%}', 'win_rate': '{:.2%}',
})

,name,regime,days,return,volatility_annual,sharpe,max_drawdown,win_rate,start,end,average_bi_rate,average_pmi,pct_pmi_expansion
16,ML_naive_majority,2020_krisis,242,8.20%,36.77%,0.29,-44.08%,52.48%,5.000000,3.750000,4.317149,42.724752,0.276860
36,Portfolio_hrp,2020_krisis,242,4.60%,35.14%,0.19,-42.19%,53.31%,5.000000,3.750000,4.317149,42.724752,0.276860
32,Portfolio_min_volatility,2020_krisis,242,1.78%,32.84%,0.09,-37.36%,52.07%,5.000000,3.750000,4.317149,42.724752,0.276860
28,Portfolio_max_sharpe,2020_krisis,242,-4.40%,39.37%,-0.03,-42.73%,49.17%,5.000000,3.750000,4.317149,42.724752,0.276860
8,Momentum_12_1,2020_krisis,242,-9.93%,36.88%,-0.23,-45.06%,49.59%,5.000000,3.750000,4.317149,42.724752,0.276860
20,ML_random_forest,2020_krisis,242,-12.06%,31.79%,-0.40,-38.96%,24.79%,5.000000,3.750000,4.317149,42.724752,0.276860
24,ML_xgboost,2020_krisis,242,-13.46%,32.36%,-0.44,-40.87%,28.51%,5.000000,3.750000,4.317149,42.724752,0.276860
0,MA_50_200,2020_krisis,242,-14.77%,32.23%,-0.49,-44.10%,35.54%,5.000000,3.750000,4.317149,42.724752,0.276860
12,ML_logistic_regression,2020_krisis,242,-20.12%,30.20%,-0.77,-40.78%,16.53%,5.000000,3.750000,4.317149,42.724752,0.276860
4,MeanReversion_BB_RSI,2020_krisis,242,-29.33%,42.39%,-0.74,-44.53%,24.38%,5.000000,3.750000,4.317149,42.724752,0.276860


## Sensitivity biaya transaksi: 0x, 1x, 2x

Semua kandidat diuji ulang dengan fee beli, fee jual, dan slippage dikalikan bersama. Perubahan CAGR, Sharpe, turnover, dan drawdown menunjukkan apakah edge cukup kuat untuk menanggung friksi transaksi.

In [7]:
cost_rows = []
cost_results = {}
for multiple in COST_MULTIPLES:
    results = run_candidates(strategy_signals, cost_multiple=multiple)
    cost_results[multiple] = results
    table = metrics_table(results, cost_multiple=multiple).reset_index()
    cost_rows.append(table.loc[table['category'].eq('strategy_or_portfolio')])
cost_sensitivity = pd.concat(cost_rows, ignore_index=True)
cost_sensitivity[['name', 'cost_multiple', 'CAGR', 'Sharpe', 'Max Drawdown', 'turnover_annual']].sort_values(['name', 'cost_multiple']).style.format({
    'CAGR': '{:.2%}', 'Sharpe': '{:.2f}', 'Max Drawdown': '{:.2%}', 'turnover_annual': '{:.2f}x',
})

,name,cost_multiple,CAGR,Sharpe,Max Drawdown,turnover_annual
0,MA_50_200,0.000000,6.14%,0.16,-45.78%,8.79x
10,MA_50_200,1.000000,3.84%,0.04,-47.00%,8.79x
20,MA_50_200,2.000000,1.58%,-0.07,-48.79%,8.79x
3,ML_logistic_regression,0.000000,1.30%,-0.24,-40.43%,9.40x
13,ML_logistic_regression,1.000000,-1.05%,-0.45,-40.78%,9.40x
23,ML_logistic_regression,2.000000,-3.36%,-0.65,-41.64%,9.39x
4,ML_naive_majority,0.000000,6.58%,0.18,-43.98%,2.51x
14,ML_naive_majority,1.000000,5.91%,0.14,-44.08%,2.51x
24,ML_naive_majority,2.000000,5.25%,0.10,-44.19%,2.51x
5,ML_random_forest,0.000000,3.05%,-0.06,-38.88%,11.54x


## Sensitivity parameter MA

Parameter grid yang diuji: 20/50, 50/200, dan 100/200. Semua konfigurasi memakai biaya 1x dan execution rule yang sama.

In [8]:
ma_grid = [(20, 50), (50, 200), (100, 200)]
ma_rows = []
for fast, slow in ma_grid:
    executed = ma_crossover_50_200(prices, fast_window=fast, slow_window=slow)
    signal = executed_to_signal(executed)
    result = backtest(signal, prices, master, config=config_for_multiple(1.0))
    row = result.metrics.loc['strategy'].to_dict()
    row.update({'fast': fast, 'slow': slow})
    ma_rows.append(row)
ma_sensitivity = pd.DataFrame(ma_rows).sort_values('CAGR', ascending=False)
ma_sensitivity.style.format({
    'CAGR': '{:.2%}', 'volatility_annual': '{:.2%}', 'Sharpe': '{:.2f}',
    'Sortino': '{:.2f}', 'Max Drawdown': '{:.2%}', 'Calmar': '{:.2f}',
    'win_rate': '{:.2%}', 'turnover_annual': '{:.2f}x',
})

,CAGR,volatility_annual,Sharpe,Sortino,Max Drawdown,Calmar,win_rate,turnover_annual,fast,slow
2,5.07%,20.14%,0.11,0.16,-49.55%,0.10,50.19%,8.01x,100,200
1,3.84%,19.62%,0.04,0.06,-47.00%,0.08,50.65%,8.79x,50,200
0,-0.14%,20.09%,-0.14,-0.20,-36.94%,-0.00,49.28%,25.01x,20,50


## Bootstrap confidence interval Sharpe Ratio

Confidence interval memakai moving-block bootstrap dengan blok 20 hari agar ketergantungan jangka pendek pada return tidak seluruhnya dihancurkan. Ini mengukur ketidakpastian sampling, bukan menghilangkan risiko regime change atau bias data.

In [9]:
def bootstrap_sharpe(returns, risk_free_daily, n_boot=2000, block_size=20, seed=RANDOM_STATE):
    aligned = pd.concat([returns.rename('return'), risk_free_daily.rename('rf')], axis=1).dropna()
    excess = (aligned['return'] - aligned['rf']).to_numpy()
    if len(excess) < block_size * 3:
        return {'sharpe': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n_obs': len(excess)}
    rng = np.random.default_rng(seed)
    n_blocks = int(np.ceil(len(excess) / block_size))
    values = []
    for _ in range(n_boot):
        starts = rng.integers(0, len(excess) - block_size + 1, size=n_blocks)
        sample = np.concatenate([excess[start:start + block_size] for start in starts])[:len(excess)]
        std = sample.std(ddof=1)
        values.append(sample.mean() / std * np.sqrt(TRADING_DAYS_PER_YEAR) if std > 0 else np.nan)
    observed_std = excess.std(ddof=1)
    observed = excess.mean() / observed_std * np.sqrt(TRADING_DAYS_PER_YEAR) if observed_std > 0 else np.nan
    values = np.asarray(values)
    return {
        'sharpe': observed, 'ci_low': float(np.nanpercentile(values, 2.5)),
        'ci_high': float(np.nanpercentile(values, 97.5)), 'n_obs': len(excess),
    }

bootstrap_rows = []
for name, result in base_results.items():
    row = bootstrap_sharpe(result.equity_curve['daily_return'], result.equity_curve['risk_free_daily'])
    row['name'] = name
    bootstrap_rows.append(row)
for benchmark in ['benchmark_lq45', 'benchmark_equal_weight']:
    first = next(iter(base_results.values())).equity_curve
    returns = first[benchmark].pct_change().fillna(0.0)
    row = bootstrap_sharpe(returns, first['risk_free_daily'])
    row['name'] = benchmark
    bootstrap_rows.append(row)
bootstrap_table = pd.DataFrame(bootstrap_rows).set_index('name')
bootstrap_table.style.format({'sharpe': '{:.2f}', 'ci_low': '{:.2f}', 'ci_high': '{:.2f}'})

,sharpe,ci_low,ci_high,n_obs
name,,,,
MA_50_200,0.04,-0.55,0.68,2410
MeanReversion_BB_RSI,-0.02,-0.57,0.57,2410
Momentum_12_1,0.04,-0.55,0.69,2410
ML_logistic_regression,-0.45,-1.10,0.16,2410
ML_naive_majority,0.14,-0.47,0.77,2410
ML_random_forest,-0.27,-0.84,0.32,2410
ML_xgboost,-0.52,-1.09,0.01,2410
Portfolio_max_sharpe,-0.15,-0.74,0.49,2410
Portfolio_min_volatility,-0.05,-0.56,0.56,2410


## Kesimpulan jujur berbasis aturan

Sel berikut menghasilkan kesimpulan otomatis dari hasil net biaya 1x. `strict beat` mensyaratkan CAGR dan Sharpe lebih tinggi daripada kedua benchmark. Bila tidak ada strategi yang memenuhi syarat, hasilnya menyatakan tidak ada bukti kuat strategi mengalahkan benchmark pada sampel out-of-sample ini.

In [10]:
benchmarks = all_metrics.loc[all_metrics['category'].eq('benchmark')]
candidates = all_metrics.loc[all_metrics['category'].eq('strategy_or_portfolio')]
best_benchmark_cagr = benchmarks['CAGR'].max()
best_benchmark_sharpe = benchmarks['Sharpe'].max()
strict_beats = candidates.loc[
    (candidates['CAGR'] > best_benchmark_cagr)
    & (candidates['Sharpe'] > best_benchmark_sharpe)
].index.tolist()

if strict_beats:
    print('Strategi yang strict beat benchmark setelah biaya 1x:', ', '.join(strict_beats))
else:
    print('Tidak ada strategi yang strict beat kedua benchmark setelah biaya 1x pada sampel out-of-sample ini.')

for name, row in candidates.iterrows():
    cost_rows_name = cost_sensitivity.loc[cost_sensitivity['name'].eq(name)].sort_values('cost_multiple')
    cagr_0x = cost_rows_name.loc[cost_rows_name['cost_multiple'].eq(0.0), 'CAGR'].iloc[0]
    cagr_2x = cost_rows_name.loc[cost_rows_name['cost_multiple'].eq(2.0), 'CAGR'].iloc[0]
    status = 'mengungguli' if name in strict_beats else 'tidak membuktikan keunggulan'
    print(f'{name}: {status}; CAGR 0x={cagr_0x:.2%}, 1x={row["CAGR"]:.2%}, 2x={cagr_2x:.2%}, Sharpe={row["Sharpe"]:.2f}.')

Tidak ada strategi yang strict beat kedua benchmark setelah biaya 1x pada sampel out-of-sample ini.
MA_50_200: tidak membuktikan keunggulan; CAGR 0x=6.14%, 1x=3.84%, 2x=1.58%, Sharpe=0.04.
MeanReversion_BB_RSI: tidak membuktikan keunggulan; CAGR 0x=8.03%, 1x=0.50%, 2x=-6.51%, Sharpe=-0.02.
Momentum_12_1: tidak membuktikan keunggulan; CAGR 0x=5.21%, 1x=3.16%, 2x=1.16%, Sharpe=0.04.
ML_logistic_regression: tidak membuktikan keunggulan; CAGR 0x=1.30%, 1x=-1.05%, 2x=-3.36%, Sharpe=-0.45.
ML_naive_majority: tidak membuktikan keunggulan; CAGR 0x=6.58%, 1x=5.91%, 2x=5.25%, Sharpe=0.14.
ML_random_forest: tidak membuktikan keunggulan; CAGR 0x=3.05%, 1x=0.11%, 2x=-2.74%, Sharpe=-0.27.
ML_xgboost: tidak membuktikan keunggulan; CAGR 0x=0.89%, 1x=-3.07%, 2x=-6.87%, Sharpe=-0.52.
Portfolio_max_sharpe: tidak membuktikan keunggulan; CAGR 0x=0.56%, 1x=-0.14%, 2x=-0.83%, Sharpe=-0.15.
Portfolio_min_volatility: tidak membuktikan keunggulan; CAGR 0x=3.40%, 1x=2.82%, 2x=2.24%, Sharpe=-0.05.
Portfolio_hrp: 

In [11]:
reports_dir = ROOT / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)
processed_dir = ROOT / 'data' / 'processed'
all_metrics.to_csv(reports_dir / 'evaluation_metrics_all.csv')
dashboard_benchmarks = all_metrics.loc[all_metrics['category'].eq('benchmark')].copy()
dashboard_benchmarks['cost_multiple'] = np.nan
dashboard_metrics = pd.concat([cost_sensitivity, dashboard_benchmarks], ignore_index=True)
dashboard_metrics.to_csv(processed_dir / 'evaluation_metrics_all.csv', index=False)
regime_table.to_csv(reports_dir / 'evaluation_regime_metrics.csv', index=False)
regime_table.to_csv(processed_dir / 'evaluation_regime_metrics.csv', index=False)
cost_sensitivity.to_csv(reports_dir / 'evaluation_cost_sensitivity.csv', index=False)
cost_sensitivity.to_csv(processed_dir / 'evaluation_cost_sensitivity.csv', index=False)
ma_sensitivity.to_csv(reports_dir / 'evaluation_ma_sensitivity.csv', index=False)
ma_sensitivity.to_csv(processed_dir / 'evaluation_ma_sensitivity.csv', index=False)
bootstrap_table.to_csv(reports_dir / 'evaluation_sharpe_bootstrap.csv')
bootstrap_table.to_csv(processed_dir / 'evaluation_sharpe_bootstrap.csv')
regime_markers.to_csv(reports_dir / 'evaluation_regime_markers.csv')
regime_markers.to_csv(processed_dir / 'evaluation_regime_markers.csv')
equity_rows = []
for multiple, result_group in cost_results.items():
    for name, result in result_group.items():
        equity_rows.append(result.equity_curve[['equity']].rename(columns={'equity': 'equity'}).assign(name=name, cost_multiple=multiple).reset_index())
benchmark_source = next(iter(base_results.values())).equity_curve
for benchmark, column in [('benchmark_lq45', 'benchmark_lq45'), ('benchmark_equal_weight', 'benchmark_equal_weight')]:
    equity_rows.append(benchmark_source[[column]].rename(columns={column: 'equity'}).assign(name=benchmark, cost_multiple=np.nan).reset_index())
pd.concat(equity_rows, ignore_index=True).to_csv(processed_dir / 'evaluation_equity.csv', index=False)
print('Seluruh tabel evaluasi disimpan ke reports/.')

Seluruh tabel evaluasi disimpan ke reports/.
